In [2]:
import datetime
from datetime import timedelta
import urllib.request
from pyspark import SparkConf
from pyspark import SparkContext
from pyspark.pandas.utils import spark_column_equals
from pyspark.sql import SparkSession, Window
import sys
from collections import namedtuple
from pyspark.sql.functions import col, countDistinct, sort_array, collect_list, array, lit 
from pyspark.sql.types import *
from pyspark.sql.functions import *
import builtins
import random
import urllib.request
import os
import time

spark = SparkSession.builder \
    .appName("MyPySparkApp") \
    .master("local[*]") \
    .getOrCreate()

    
sc = spark.sparkContext
 
start_time = time.time()
# Verify the session is active
print(f"Spark version: {spark.version}")

/home/aries/sparky/.venv/lib/python3.11/site-packages/pyspark/pandas/__init__.py:33: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommended to use pandas < 3.0.0 for now.
  require_minimum_pandas_version()
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
26/11/24 21:27:40 WARN Utils: Your hostname, Aries, resolves to a loopback address: 127.0.1.1; using 10.255.255.254 instead (on interface lo)
26/11/24 21:27:40 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
/home/aries/sparky/.venv/lib/python3.11/site-packages/pyspark/testing/utils.py:127: FutureWarning: PySpark does not yet fully support pandas >= 3.0.0. Some features may not work correctly. It is recommend

Spark version: 4.2.0


In [ ]:
x = [
("214252",	"111","Am considering taking Tesla private at $420. Funding secured.",	    "12/30/2021 00:00:00"),
("214263",	"113","Am considering taking Tesla private at $420. Funding secured.",	    "12/30/2021 00:00:00"),
("214257",	"119","Am considering taking Tesla private at $420. Funding secured.",	    "12/30/2021 00:00:00"),
("214255",	"116","Am considering taking Tesla private at $420. Funding secured.",	    "12/30/2021 00:00:00"),
("739252",	"111","Despite the constant negative press covfefe",	                     "01/01/2022 00:00:00"),
("846402",	"111","Following @NickSinghTech on Twitter changed my life!",                "02/14/2022 00:00:00"),
("241425",	"254","If the salary is so competitive why won’t you tell me what it is?",   "03/01/2022 00:00:00"),
("231574",	"148","I no longer have a manager. I can't be managed",                      "03/23/2022 00:00:00"),
]
df_daily_sales = spark.createDataFrame(x, ['tweet_id', 'user_id', 'msg', 'tweet_date'])

# df_daily_sales.show(truncate=False)

df_with_timestamp = df_daily_sales.withColumn("tweet_date", to_timestamp(col("tweet_date"), "MM/dd/yyyy HH:mm:ss"))

# df_with_timestamp.show(truncate=False)

df_with_timestamp.show(truncate=False)

+--------+-------+-----------------------------------------------------------------+-------------------+
|tweet_id|user_id|msg                                                              |tweet_date         |
+--------+-------+-----------------------------------------------------------------+-------------------+
|214252  |111    |Am considering taking Tesla private at $420. Funding secured.    |2021-12-30 00:00:00|
|214263  |113    |Am considering taking Tesla private at $420. Funding secured.    |2021-12-30 00:00:00|
|214257  |119    |Am considering taking Tesla private at $420. Funding secured.    |2021-12-30 00:00:00|
|214255  |116    |Am considering taking Tesla private at $420. Funding secured.    |2021-12-30 00:00:00|
|739252  |111    |Despite the constant negative press covfefe                      |2022-01-01 00:00:00|
|846402  |111    |Following @NickSinghTech on Twitter changed my life!             |2022-02-14 00:00:00|
|241425  |254    |If the salary is so competitive why w

26/11/25 00:17:43 WARN HeartbeatReceiver: Removing executor driver with no recent heartbeats: 929393 ms exceeds timeout 120000 ms
26/11/25 00:17:43 WARN SparkContext: Killing executors is not supported by current scheduler.
26/11/25 00:17:43 WARN Executor: Issue communicating with driver in heartbeater
org.apache.spark.SparkException: Exception thrown in awaitResult: 
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:70)
	at org.apache.spark.util.SparkThreadUtils$.awaitResult(SparkThreadUtils.scala:44)
	at org.apache.spark.util.ThreadUtils$.awaitResult(ThreadUtils.scala:359)
	at org.apache.spark.rpc.RpcTimeout.awaitResult(RpcTimeout.scala:75)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:101)
	at org.apache.spark.rpc.RpcEndpointRef.askSync(RpcEndpointRef.scala:85)
	at org.apache.spark.storage.BlockManagerMaster.registerBlockManager(BlockManagerMaster.scala:85)
	at org.apache.spark.storage.BlockManager.reregister(BlockManager.scala:707

In [ ]:
columns = ["candidate_id",	"skill"]
data = [
    ("123",	"Python"),
    ("123",	"Tableau"),
    ("123",	"PostgreSQL"),
    ("234",	"R"),
    ("234",	"PowerBI"),
    ("234",	"SQL Server"),
    ("345",	"Python"),
    ("345",	"Tableau"),
]
df = spark.createDataFrame(data, columns)
# df.show()
# df.printSchema()
# using dictionary to check for the skills needed
required_skills_dict = {"Python", "Tableau", "PostgreSQL"}
filtered_candidates = df.filter(df.skill.isin(required_skills_dict))
# filtered_candidates.show()
candidates_with_skills = (filtered_candidates
                          .groupBy("candidate_id")
                          .agg(
                              countDistinct("skill").alias("skill_count"),
                              collect_list("skill").alias("skills")
                            )
                          .filter(col("skill_count") == len(required_skills_dict))
                          .orderBy("candidate_id"))
print("This is the candidates with skills")
candidates_with_skills.show(truncate=False)

filtered_candidates.createOrReplaceTempView("all_candidates")
spark.sql("""
       select
       candidate_id,
       collect_list(skill) as skills
       from all_candidates
       where
       skill in ("Python", "Tableau", "PostgreSQL")
       group by candidate_id
       having count(distinct skill) >=3
       order by candidate_id;   
        """).show(truncate=False)

In [ ]:
columns1 = ["page_id",	"page_name"]
data1 = [
    ("20001",	"SQL Solutions"),
    ("20045",	"Brain Exercises"),
    ("20701",	"Tips for Data Analysts")
]
columns2=["user_id",	"page_id",	"liked_date"]
data2= [
    ("111",	"20001",	"04/08/2022 00:00:00"),
    ("121",	"20045",	"03/12/2022 00:00:00"),
    ("156",	"20001",	"07/25/2022 00:00:00")
]

df1 = spark.createDataFrame(data1, columns1)
df2 = spark.createDataFrame(data2, columns2)

df1.join(df2, "page_id", "anti").show(truncate=False)
# df1.join(df2, "page_id", "anti").groupby("page_id").agg(count("user_id").asc()).show(truncate=False)
df1.join(df2, "page_id", "left").filter(col("user_id").isNull()).show(truncate=False)
df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")
spark.sql("""
   select *
   from df1 as p
   left join df2 as l
   on p.page_id=l.page_id
   where l.page_id is NULL
    """).show(truncate=False)


In [ ]:
columns = ["user_id",	"device_type","view_time"]
data = [
    ("123",	"tablet",	"01/02/2022 00:00:00"),
    ("125",	"laptop",	"01/07/2022 00:00:0"),
    ("128",	"laptop",	"02/09/2022 00:00:0"),
    ("129",	"phone",	"02/09/2022 00:00:0"),
    ("145",	"tablet",	"02/24/2022 00:00:0"),
]

df = spark.createDataFrame(data, columns)
df.show()

df.createOrReplaceTempView("df")

spark.sql("""
    select
        sum(case when device_type='laptop' then 1 else 0 end) as laptop_views,
        sum(case when device_type in ('mobile','tablet') then 1 else 0 end) as mobile_views
    from df;
    """).show(truncate=False)


df.agg(
    sum(when(col("device_type")=='laptop',1).otherwise(0)).alias("laptop_views"),
    sum(when(col("device_type").isin('mobile','tablet'),1).otherwise(0)).alias("mobile_views")
).show()

In [ ]:
columns = ["user_id",	"post_id",	"post_content",	"post_date"]
data = [
    ("151652",	"599415",	"Need a hug",	"07/10/2021 12:00:00"),
    ("661093",	"624356",	"Bed. Class 8-12. Work 12-3. Gym 3-5 or 6. Then class 6-10. Another day that's gonna fly by. I miss my girlfriend",	"07/29/2021 13:00:00"),
    ("004239",	"784254",	"Happy 4th of July!",	"07/04/2021 11:00:00"),
    ("661093",	"442560",	"Just going to cry myself to sleep after watching Marley and Me.",	"07/08/2021 14:00:00"),
    ("151652",	"111766",	"I'm so done with covid - need travelling ASAP!",	"07/12/2021 19:00:00")
]

df = spark.createDataFrame(data, columns)
df = df.withColumn("post_date", to_timestamp("post_date", "MM/dd/yyyy HH:mm:ss"))

df.createOrReplaceTempView("df")
spark.sql(""" 
   select
        user_id,
        date_diff(max(post_date), min(post_date)) as days_diff
    from
        df
    where 
        year(post_date)=2021
    group by 
        user_id
    order by days_diff desc       
    """).show(truncate=False)

( df
    .filter(year('post_date')==2021)
    .groupBy("user_id")
    .agg(date_diff(max("post_date"),min("post_date")).alias("days_difference"))
    .filter("count(post_id)>1")
    .orderBy(col("days_difference").desc())
    .show() 
)

In [ ]:
columns=["message_id",	"sender_id",	"receiver_id",	"content",	"sent_date"]
data = [
    ("901",	"3601",	"4500",	"You up?",	"08/03/2022 00:00:00"),
    ("902",	"4500",	"3601",	"Only if you're buying",	"08/03/2022 00:00:00"),
    ("743",	"3601",	"8752",	"Let's take this offline",	"06/14/2022 00:00:00"),
    ("922",	"3601",	"4500",	"Get on the call",	"08/10/2022 00:00:00"),
]

rdd = spark.sparkContext.parallelize(data)
df = rdd.toDF(columns)
# df = spark.createDataFrame(data, columns)
df = df.withColumn("sent_date", to_timestamp("sent_date", "MM/dd/yyyy HH:mm:ss"))

df.createOrReplaceTempView("df")

(
    df.filter((year("sent_date")=="2022") & (month("sent_date")==8))
    .groupby("sender_id")
    .count()
    .orderBy(col("count").desc())
    .limit(2)
).show(truncate=False)

In [ ]:
columns = ["job_id",	"company_id",	"title",	"description"]

data = [
(
    "248",	
    "827",	
    "Business Analyst",	
    """Business analyst evaluates past and current business data with the primary 
    goal of improving decision-making processes within organizations."""
),
(
    "149",	
    "845",  
    "Business Analyst",	
    """Business analyst evaluates past and current business data with the primary 
    goal of improving decision-making processes within organizations."""
),
(
    "945",	
    "345",	
    "Data Analyst",     
    """Data analyst reviews data to identify key insights into a business's customers 
    and ways the data can be used to solve problems."""
),
(
    "164",	
    "345",	
    "Data Analyst",     
    """Data analyst reviews data to identify key insights into a business's customers 
    and ways the data can be used to solve problems."""
),
(
    "172",	
    "244",  
    "Data Engineer",    
    """Data engineer works in a variety of settings to build systems that collect, 
    manage, and convert raw data into usable information for data scientists and 
    business analysts to interpret."""
),
]

rdd = sc.parallelize(data)
df = rdd.toDF(columns)
df.createOrReplaceTempView("df")
spark.sql("""
         select
            company_id,
            title,
            description,
            count(job_id) as job_count
        from df
        group by company_id, title, description 
    """).show(truncate=False)

In [ ]:
columns1=["order_id",	"user_id",	"quantity",	"status",	"date",	"price"]
data1=[
	("100101",	"111",	"10",	"Cancelled",	"08/17/2022 12:00:00",	"9.80"),
	("100102",	"111",  "10","Completed",	"08/17/2022 12:00:00",	"10.00"),
	("100259",	"148",  "35","Completed","08/25/2022 12:00:00",	"5.10"),
	("100264",	"148",  "40","Completed","08/26/2022 12:00:00","4.80"),
	("100305",	"300",  "15","Completed","09/05/2022 12:00:00","10.00"),
	("100400",	"178",  "32","Completed","09/17/2022 12:00:00","12.00"),
	("100565",	"265",  "2", "Completed","09/27/2022 12:00:00","8.70"),
]


columns2 = ["user_id",	"city",	"email",	"signup_date"]
data2 = [
	("111",	"San Francisco",	"rrok10@gmail.com",	"08/03/2021 12:00:00"),
	("148",	"Boston",	"sailor9820@gmail.com",	"08/20/2021 12:00:00"),
	("178",	"San Francisco",	"harrypotterfan182@gmail.com",	"01/05/2022 12:00:00"),
	("265",	"Denver",	"shadower_@hotmail.com",	"02/26/2022 12:00:00"),
	("300",	"San Francisco",	"houstoncowboy1122@hotmail.com",	"06/30/2022 12:00:00"),
]
rdd1 = sc.parallelize(data1)
rdd2 = sc.parallelize(data2)
df1 = rdd1.toDF(columns1)
df2 = rdd2.toDF(columns2)


df1.createOrReplaceTempView("df1")
df2.createOrReplaceTempView("df2")
spark.sql(""" 
          select 
                c.city,
                count(*) as count_per_city
            from df1 as t
            join df2 as c
            on c.user_id=t.user_id
            where t.status='Completed'
            group by c.city
            order by count_per_city desc;         
""").show(truncate=False)



In [ ]:
columns= [ "review_id",	"user_id",	"submit_date",	"product_id",	"stars" ]
data = [
	( "6171",	"123",	"06/08/2022 00:00:00",	"50001",	"4" ),
	( "7802",	"265",	"06/10/2022 00:00:00",	"69852","4" ),
	( "5293",	"362",	"06/18/2022 00:00:00",	"50001","3" ),
	( "6352",	"192",	"07/26/2022 00:00:00",	"69852","3"),
	( '4517',	"981",	"07/05/2022 00:00:00",	"69852","2" ),
]

rdd = sc.parallelize(data)
df = rdd.toDF(columns)
df = df.withColumn("submit_date", to_timestamp("submit_date", "MM/dd/yyyy HH:mm:ss"))
# df.show()
# df.printSchema()
df.createOrReplaceTempView("df")

spark.sql(''' 
    select
        month(submit_date) as months,
        product_id,
        avg(stars) as average
    from    
        df
    group by 
        months,
        product_id
    order by months desc, product_id desc;
    ''').show(truncate=False)

In [ ]:
columns = [ "app_id",	"event_type",	"timestamp" ]
data = [
	( "123",	"impression",	"07/18/2022 11:36:12" ),
	( "123",	"impression",	"07/18/2022 11:37:12" ),
	( "123",	"click",	"07/18/2022 11:37:42" ),
	( "234",	"impression",	"07/18/2022 14:15:12" ),
	( "234",	"click",	"07/18/2022 14:16:12" ),
]

rdd = sc.parallelize(data)
df = rdd.toDF(columns)
df = df.withColumn("timestamp", to_timestamp("timestamp", "MM/dd/yyyy HH:mm:ss"))

df.createOrReplaceTempView("df")
spark.sql(''' 
    select
    app_id,
    100.0 *
    sum(case when event_type='click' then 1 else 0 end)
    /
    sum(case when event_type='impression' then 1 else 0 end)
    as click_through_rate
    from df     
    group by app_id  
    ''').show(truncate=False)

In [ ]:
cols1 = [ "employee_id",	"query_id",	"query_starttime",	"execution_time" ]
data1 = [
	( "3",	"856987",	"07/01/2023 03:25:12",	"2698" ),
	( "3",	"286115",	"07/01/2023 04:34:38",	"2705" ),
	( "3",	"33683","07/02/2023 10:55:14",	"91" ),
	( "1",	"413477","07/15/2023 11:35:09",	"470" ),
	( "1", "421983", "07/01/2023 14:33:47", "3020" ),
	( "2",	"17745","07/01/2023 14:33:47",	"2093" ),
	( "2", "958745", "07/02/2023 08:11:45", "512" ),
	( "2", "684293", "07/22/2023 18:42:31", "1630" ),
	( "2", "385739", "07/25/2023 14:25:17", "240" ),
	( "2", "123456", "07/26/2023 16:12:18", "950" ),
]


cols2 = [ "employee_id",	"full_name",	"gender" ]
data2 = [
	( "1",	"Judas Beardon",	"Male" ),
	( "2",	"Lainey Franciotti",	"Female" ),
	( "3",	"Ashbey Strahan",	"Male" ),
	( "4",	"Nikhil",	"Male" ),
]
rdd1 = sc.parallelize(data1)
rdd2 = sc.parallelize(data2)
df1 = rdd1.toDF(cols1)
df2 = rdd2.toDF(cols2)
df1 = df1.withColumn("query_starttime", to_timestamp("query_starttime","MM/dd/yyyy HH:mm:ss"))
# df1.show()
# df2.show()
df2.createOrReplaceTempView("employees")
df1.createOrReplaceTempView("queries")
spark.sql("""
  SELECT
    e.employee_id,
    COALESCE(COUNT(DISTINCT q.query_id), 0) AS unique_queries
  FROM employees AS e
  LEFT JOIN queries AS q
    ON e.employee_id = q.employee_id
      AND q.query_starttime >= '2023-07-01T00:00:00Z'
      AND q.query_starttime < '2023-10-01T00:00:00Z'
  GROUP BY e.employee_id
""").show(truncate=False)